In [1]:
import os
import cv2
import numpy as np
from skimage.feature import hog
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, f1_score
import time

In [2]:
# --- CONFIGURATION ---
# Change this to the path where you saved your Kaggle images
DATASET_PATH = r"Dataset\\asl_alphabet_train"
MAX_IMAGES_PER_CLASS = 50 # The "portion" of data to keep it fast
IMAGE_SIZE = (64, 64) # Downsampled resolution

baseline_features = [] # For k-NN
proposed_features = [] # For HOG-SVM
labels = []

print(f"Loading up to {MAX_IMAGES_PER_CLASS} images per class...")

# Loop through each letter/number folder
for class_name in os.listdir(DATASET_PATH):
    class_folder = os.path.join(DATASET_PATH, class_name)
    
    if not os.path.isdir(class_folder):
        continue
        
    count = 0
    for image_name in os.listdir(class_folder):
        if count >= MAX_IMAGES_PER_CLASS:
            break
            
        img_path = os.path.join(class_folder, image_name)
        img = cv2.imread(img_path)
        
        if img is not None:
            # 1. Convert to grayscale as specified in proposal
            gray_img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
            
            # 2. Downsample
            resized_img = cv2.resize(gray_img, IMAGE_SIZE)
            
            # --- BASELINE EXTRACTION (Naive Pixels) ---
            flattened_pixels = resized_img.flatten()
            baseline_features.append(flattened_pixels)
            
            # --- PROPOSED EXTRACTION (HOG) ---
            features_hog = hog(resized_img, 
                               orientations=9, 
                               pixels_per_cell=(8, 8),
                               cells_per_block=(2, 2), 
                               block_norm='L2-Hys', 
                               visualize=False)
            proposed_features.append(features_hog)
            
            labels.append(class_name.upper())
            count += 1

# Convert lists to NumPy arrays for scikit-learn
X_baseline = np.array(baseline_features)
X_proposed = np.array(proposed_features)
y = np.array(labels)

# Split the data (80% training, 20% testing)
X_base_train, X_base_test, y_train, y_test = train_test_split(X_baseline, y, test_size=0.2, random_state=42)
X_prop_train, X_prop_test, _, _ = train_test_split(X_proposed, y, test_size=0.2, random_state=42)

print(f"Extraction complete! Total images loaded: {len(y)}")
print(f"Baseline Feature Shape (Pixels): {X_baseline.shape[1]}")
print(f"Proposed Feature Shape (HOG): {X_proposed.shape[1]}")

Loading up to 50 images per class...
Extraction complete! Total images loaded: 1450
Baseline Feature Shape (Pixels): 4096
Proposed Feature Shape (HOG): 1764


In [3]:
print("--- Training Baseline Model (k-NN) ---")
start_time = time.time()

# Basic k-NN applied directly to flattened pixels
knn_baseline = KNeighborsClassifier(n_neighbors=3)
knn_baseline.fit(X_base_train, y_train)

# Predict and Evaluate
base_predictions = knn_baseline.predict(X_base_test)
base_f1 = f1_score(y_test, base_predictions, average='weighted')

print(f"Baseline Training Time: {time.time() - start_time:.2f} seconds")
print(f"Baseline F1-Score: {base_f1:.4f}")
print("\nBaseline Classification Report:")
print(classification_report(y_test, base_predictions))

--- Training Baseline Model (k-NN) ---
Baseline Training Time: 0.11 seconds
Baseline F1-Score: 0.8748

Baseline Classification Report:
              precision    recall  f1-score   support

           A       0.89      1.00      0.94         8
           B       0.69      0.92      0.79        12
           C       1.00      0.86      0.92         7
           D       0.86      0.86      0.86         7
         DEL       0.79      1.00      0.88        11
           E       0.58      0.70      0.64        10
           F       0.82      1.00      0.90         9
           G       1.00      0.83      0.91        12
           H       1.00      0.92      0.96        12
           I       0.71      0.71      0.71         7
           J       0.60      1.00      0.75         9
           K       1.00      0.94      0.97        17
           L       0.93      1.00      0.97        14
           M       1.00      0.88      0.93         8
           N       1.00      0.70      0.82        10


In [4]:
print("--- Training Proposed Model (HOG-SVM) ---")
start_time = time.time()

# The Proposed Pipeline
proposed_pipeline = Pipeline([
    ('scaler', StandardScaler()), # Essential for SVM math
    ('pca', PCA(n_components=0.95)), # Retain 95% variance
    ('svm', SVC(kernel='rbf', C=1.0, gamma='scale')) # RBF Kernel
])

proposed_pipeline.fit(X_prop_train, y_train)

# Predict and Evaluate
prop_predictions = proposed_pipeline.fit(X_prop_train, y_train).predict(X_prop_test)
prop_f1 = f1_score(y_test, prop_predictions, average='weighted')

print(f"Proposed Training Time: {time.time() - start_time:.2f} seconds")
print(f"Proposed F1-Score: {prop_f1:.4f}")
print("\nProposed Classification Report:")
print(classification_report(y_test, prop_predictions))

print("-" * 30)
print(f"IMPROVEMENT: The Proposed model beat the baseline by {(prop_f1 - base_f1):.4f} in F1-Score!")

--- Training Proposed Model (HOG-SVM) ---
Proposed Training Time: 2.82 seconds
Proposed F1-Score: 0.9016

Proposed Classification Report:
              precision    recall  f1-score   support

           A       1.00      0.88      0.93         8
           B       0.92      0.92      0.92        12
           C       1.00      1.00      1.00         7
           D       1.00      0.86      0.92         7
         DEL       1.00      1.00      1.00        11
           E       0.50      0.90      0.64        10
           F       1.00      1.00      1.00         9
           G       1.00      0.83      0.91        12
           H       1.00      0.92      0.96        12
           I       0.80      0.57      0.67         7
           J       1.00      1.00      1.00         9
           K       1.00      0.88      0.94        17
           L       1.00      1.00      1.00        14
           M       1.00      0.88      0.93         8
           N       1.00      0.70      0.82        